# 🧬 BioEmu Tools Example

This notebook demonstrates how to sample protein conformational ensembles with **BioEmu**.

## 📖 What is BioEmu?

BioEmu is a model for fast conformational ensemble sampling from protein sequence inputs.

### Key Features:
- 🧪 **Ensemble sampling** - Generate many backbone conformations from one sequence
- 📦 **Batch processing** - Process multiple complexes in one call
- ⚡ **Flexible runtime** - Uses the cloud runtime or local standalone execution via `EnvManager`

## 📦 Imports


## Input/Output Schema

### `StructurePredictionComplex`
Represents a single biological complex for structure prediction.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Union[str, Chain, dict]]` | Chains in the complex. Each can be a sequence string, a `Chain` object, or a dict with `sequence`, `entity_type`, and `modifications` fields. |

BioEmu only supports single-chain protein monomers (one protein chain per complex).

### `BioEmuInput`
Extends `StructurePredictionInput`. Validates that each complex has exactly one protein chain with standard amino acids.

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | Protein complexes to sample. Each must contain exactly one protein chain. |

### `BioEmuConfig`
Extends `StructurePredictionConfig` with BioEmu-specific parameters.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_samples` | `int` | `500` | Number of conformations to sample per sequence |
| `model_name` | `Literal["bioemu-v1.0", "bioemu-v1.1"]` | `"bioemu-v1.1"` | BioEmu model variant to use |
| `filter_samples` | `bool` | `True` | Whether to filter generated samples using BioEmu quality checks |
| `batch_size` | `int` | `10` | Batch size control for BioEmu internal sampling |
| `output_dir` | `Optional[str]` | `None` | Optional directory for raw BioEmu output files |
| `device` | `str` | `"cuda"` | Device to run the model on (inherited from `StructurePredictionConfig`) |
| `verbose` | `bool` | `False` | Whether to print status messages (inherited) |
| `keep_on_gpu` | `bool` | `False` | Whether to keep model loaded on device after inference (inherited) |

### `BioEmuOutput`
| Field | Type | Description |
|-------|------|-------------|
| `ensembles` | `List[ProteinStructureEnsemble]` | Generated ensembles, one per input complex |

Each `ProteinStructureEnsemble` contains:

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[ProteinStructure]` | List of sampled conformational structures |
| `sequence` | `str` | The input protein sequence |

Export formats: `"pdb"` (directory tree of PDB files), `"json"` (single JSON file).

In [1]:
from bio_programming.bio_tools.tools.structure_prediction.shared_data_models import StructurePredictionComplex
from bio_programming.bio_tools.tools.structure_dynamics.bioemu import (
    BioEmuInput,
    BioEmuConfig,
    run_bioemu,
)

## 🧪 1. Sample a Single Ensemble

Generate conformations for one protein monomer.

### Configuration options:
- 🧠 **`num_samples`** - Number of conformations per sequence
- 🧬 **`model_name`** - BioEmu model variant
- 🧹 **`filter_samples`** - Whether to apply sample filtering
- 📦 **`batch_size`** - BioEmu internal batch-size control


In [2]:
complex_ = StructurePredictionComplex(
    chains=[{"sequence": "MVLSPADKTNVKAAW", "entity_type": "protein"}]
)

inputs = BioEmuInput(complexes=[complex_])
config = BioEmuConfig(num_samples=50, model_name="bioemu-v1.1", filter_samples=True)

single_result = run_bioemu(inputs, config)
print(f"Ensembles: {len(single_result.ensembles)}")
print(f"Conformations: {len(single_result.ensembles[0].structures)}")


Setting up BioEmu standalone environment...
Installing uv package manager...
  Using cached uv-0.10.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached uv-0.10.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (22.8 MB)
Installing dependencies from requirements.txt...
BioEmu setup complete!
Ensembles: 1
Conformations: 49


## 📚 2. Batch Sampling Across Complexes

Submit multiple complexes in one call and receive one ensemble per input complex.


In [3]:
complex_a = StructurePredictionComplex(
    chains=[{"sequence": "MVLSPADKTNVKAAW", "entity_type": "protein"}]
)
complex_b = StructurePredictionComplex(
    chains=[{"sequence": "GSGSGSGSGSGS", "entity_type": "protein"}]
)

batch_inputs = BioEmuInput(complexes=[complex_a, complex_b])
batch_config = BioEmuConfig(num_samples=20, verbose=False)
batch_result = run_bioemu(batch_inputs, batch_config)

for idx, ensemble in enumerate(batch_result.ensembles):
    print(f"Complex {idx}: {len(ensemble.structures)} conformations")


Complex 0: 19 conformations
Complex 1: 17 conformations


## 💾 3. Export Results

Save ensemble outputs for downstream analysis.

### Supported formats:
- 📁 **PDB directory tree** (`pdb`)
- 📄 **JSON** (`json`)


In [4]:
batch_result.export("example_output/bioemu_batch", file_format="json")
print("Batch ensembles exported to example_output/bioemu_batch.json")


Batch ensembles exported to example_output/bioemu_batch.json
